In [ ]:
#!pip install gensim matplotlib scikit-learn transformers torch requests langchain-core langchain-cohere

# Program 1
Explore pre-trained word vectors. Explore word relationships using vector arithmetic. Perform arithmetic operations and analyze results.

In [ ]:
#!pip install gensim
import gensim.downloader as api
model = api.load("glove-wiki-gigaword-50")
print("Vocabulary Size:", len(model.key_to_index))
print("\nTop 2 words similar to 'king': ",model.most_similar("king", topn=2))
print("\nResult of king - man + woman:", model.most_similar(positive=["king", "woman"],negative=["man"],topn=1))
king = model["king"]
man = model["man"]
woman = model["woman"]
print("\nManual Vector Arithmetic Result: ",model.similar_by_vector((king-man+woman), topn=2))
#22.2S

# Program 2
Use dimensionality reduction (e.g., PCA or t-SNE) to visualize word embeddings for Q 1. Select 10 words from a specific domain (e.g., sports, technology) and visualize their embeddings. Analyze clusters and relationships. Generate contextually rich outputs using embeddings. Write a program to generate 5 semantically similar words for a given input.

In [ ]:
import gensim.downloader as api
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

m = api.load("glove-wiki-gigaword-50")
words = ["cricket","football","tennis","bat","ball",
         "stadium","player","coach","match","tournament"]

p = PCA(2).fit_transform([m[w] for w in words])

for i,w in enumerate(words):
    plt.scatter(p[i,0],p[i,1])
    plt.text(p[i,0],p[i,1],w)

plt.title("Sports Word Embeddings")
plt.show()

#word = input("Enter a word: ")
word="ball"
similar = m.most_similar(word, topn=5)

print("\nTop 5 Similar Words: ",similar)
# for w,s in similar:
#     print(f"{w} : {s:.2f}")

print("\nContextual Output:")
print(word,"is related to",[w for w,s in similar])
#22.0S

# Program 3
Train a custom Word2Vec model on a small dataset. Train embeddings on a domain-specific corpus (e.g., legal, medical) and analyze how embeddings capture domain-specific semantics.

In [ ]:
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

sentences = [
    ["doctor","treats","patient"],
    ["nurse","assists","doctor"],
    ["patient","takes","medicine"],
    ["surgeon","performs","surgery"],
    ["doctor","diagnoses","disease"],
    ["hospital","treats","patient"],
    ["doctor","prescribes","medicine"]
]

model = Word2Vec(sentences, vector_size=50, window=3, min_count=1)

print("Top 5 words similar to doctor:")
for w,s in model.wv.most_similar("doctor", topn=5):
    print(f"{w} : {s:.2f}")

words = list(model.wv.index_to_key)
points = PCA(2).fit_transform(model.wv[words])

for i,w in enumerate(words):
    plt.scatter(points[i,0], points[i,1])
    plt.text(points[i,0], points[i,1], w)

plt.title("Medical Word Embeddings")
plt.show()

# Program 4
Use word embeddings to improve prompts for Generative AI model. Retrieve similar words using word embeddings. Use the similar words to enrich a GenAI prompt. Use the AI model to generate responses for the original and enriched prompts. Compare the outputs in terms of detail and relevance.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import gensim.downloader as api
from transformers import pipeline

m=api.load("glove-wiki-gigaword-50")
g=pipeline("text-generation",model="gpt2")

p="cricket"

print("Original Output:\n")
print(g(p)[0]["generated_text"])

ep=p+" including "+", ".join(w for w,_ in m.most_similar("cricket",topn=5))

print("\nEnriched Output:\n")
print(g(ep)[0]["generated_text"])

print("\nComparison:")
print("The enriched prompt generates a more detailed and relevant response because it contains additional context from word embeddings.")

# Program 5
Use word embeddings to create meaningful sentences for creative tasks. Retrieve similar words for a seed word. Create a sentence or story using these words as a starting point. Write a program that: Takes a seed word. Generates similar words. Constructs a short paragraph using these words.

In [ ]:
import gensim.downloader as api
from transformers import pipeline

m = api.load("glove-wiki-gigaword-50")
g = pipeline("text-generation", model="gpt2")

seed = "cricket" #input("Enter seed word: ")

words = [w for w,_ in m.most_similar(seed, topn=5)]

print("\nSimilar Words:",words)

prompt = seed + " related to " + ", ".join(words) + "."

print("\nGenerated Paragraph:\n")
print(g(prompt)[0]["generated_text"])

# Program 6
Use a pre-trained Hugging Face model to analyze sentiment in text. Assume a real-world application. Load the sentiment analysis pipeline. Analyze the sentiment by giving sentences as input.

In [ ]:
from transformers import pipeline
sentiment_analyzer = pipeline("sentiment-analysis")
sentences = ["I love how easy this app is to use.","The delivery was late","The movie was terrible"]
for sentence, result in zip(sentences, sentiment_analyzer(sentences)):
    print(f"Text: {sentence}")
    print(f"Sentiment: {result['label']}, Confidence: {result['score']:.4f}\n")
#1.4s

In [ ]:
from transformers import pipeline

s = pipeline("sentiment-analysis")

for i in range(3):
    text = input("Enter sentence: ")
    print(f"sentence: {text}: {s(text)}")

# Program 7
Summarize long texts using a pre-trained summarization model using Hugging Face model. Load the summarization pipeline. Take a passage as input and obtain the summarized text.

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="gpt2")

text = """
The explosion of online shopping has transformed commerce.
Consumers now expect fast delivery, transparent reviews,
and personalized recommendations. Retailers invest in
logistics and data to meet these expectations while
balancing cost and sustainability.
"""

print(summarizer("summarize: " + text)[0]["summary_text"])
#2.6s

# Program 8
Install langchain, cohere (for key), langchain-community. Get the api key (By logging into Cohere and obtaining the cohere key). Load a text document from your google drive. Create a prompt template to display the output in a particular manner.

In [ ]:
import requests
from langchain_core.prompts import PromptTemplate
from langchain_cohere import ChatCohere 

url = "https://drive.google.com/uc?id=1DGVYAgcOz3KZehzj6S4uOcJAGo4oj57G&export=download"
text = requests.get(url).text

llm = ChatCohere(cohere_api_key="XQAbXr2w15Qqe0D5Pzwx5hooGkBoXXmrJMMdxuCO",model="command-a-03-2025")

prompt = PromptTemplate(input_variables=["data"],template="Summarize the following document in 5 key points with each point not more that 12 words:\n\n{data}")

print(llm.invoke(prompt.format(data=text)).content)
#12.4s